# NB03 — Airbnb Neighbourhoods ETL

**Project:** Melbourne Airbnb End-to-End Analytics  
**Technologies:** Microsoft Fabric | PySpark | Delta Lake  
**Data Layer:** Raw CSV → Bronze → Silver → Gold (Downstream)

## 1. Objective

Clean and standardise Melbourne Airbnb neighbourhood data to support **LGA-level analysis** and consistent joins with Airbnb listings.

The validated dataset provides a geographic reference for the downstream Gold Snowflake Schema.

## 2. Data Sources & Outputs

| Type | Source |
|---|---|
| Raw Input | `Files/neighbourhoods.csv` |
| Bronze Output | `dbo.bronze_neighbourhoods` |
| Silver Output | `dbo.silver_neighbourhoods` |
| Downstream Gold | `dbo.dim_neighbourhood` |

**Data Grain:** One record per unique Local Government Area (LGA).

## 3. ETL Process

1. **Data Ingestion:** Read the Inside Airbnb Neighbourhoods CSV into the Bronze Layer.
2. **Data Cleaning:** Inspect missing values and standardise LGA names.
3. **Key Transformation:** Rename `neighbourhood` to `lga` and create a normalised `lga_key`.
4. **Data Quality Validation:** Check missing values, duplicate LGAs and key uniqueness.
5. **Silver Storage:** Save the validated dataset to `dbo.silver_neighbourhoods` in Delta format.

## 4. LGA Standardisation

The `lga_key` column is created by:

- Removing leading and trailing spaces
- Replacing multiple spaces with a single space
- Converting text to lowercase

**Example:**

| Original Value | Standardised `lga_key` |
|---|---|
| `"Melbourne"` | `melbourne` |
| `" MELBOURNE "` | `melbourne` |
| `"melbourne"` | `melbourne` |

**Design Decision:** Keep the original LGA name for reporting and use `lga_key` for consistent table joins.

This reduces JOIN mismatches caused by differences in capitalisation and whitespace.

## 5. Validated Output

| Metric | Result |
|---|---:|
| Unique LGAs | 30 |
| Silver Output | `dbo.silver_neighbourhoods` |
| Join Key | `lga_key` |

The Silver table provides a consistent geographic reference for the 30 LGAs in the September 2026 dataset.

## 6. Downstream Gold Layer

`NB04_Gold_SnowFlake_Schema` uses the validated Neighbourhoods data to create `dbo.dim_neighbourhood`.

The dimension assigns a unique `neighbourhood_id` to each LGA and connects to `dim_listing` through this key.

This supports comparisons of listing supply, room types, indicative prices and advertised availability across Melbourne's LGAs.

Cross-table relationship validation is performed in NB04.

## 7. Pipeline Dependency

**NB03 Neighbourhoods → NB04 Gold Snowflake Schema**

Neighbourhoods ETL runs independently of Listings and Calendar ETL.

The Gold ETL starts only after both the Neighbourhoods and Calendar ETL activities have succeeded, ensuring all required Silver datasets are available.

In [ ]:
# Import Required Libraries
from pyspark.sql import functions as F

In [ ]:

#Ingest Neighbourhood CSV into Bronze

raw_csv = (
    spark.read
    .option("header", "true")
    .csv("Files/neighbourhoods.csv")
)

# Validate source columns
required_columns = {
    "neighbourhood",
    "neighbourhood_group"
}

assert required_columns.issubset(
    set(raw_csv.columns)
), "Missing required source columns!"

assert raw_csv.count() > 0, "Empty Neighbourhood CSV!"

# Save raw data to Bronze
(
    raw_csv.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.bronze_neighbourhoods")
)

print("Bronze Neighbourhoods updated!")


StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 16, Finished, Available, Finished, False)

Bronze Neighbourhoods updated!


In [15]:

# Read Bronze Neighbourhoods
raw = spark.table("dbo.bronze_neighbourhoods")

# Check data structure
print("Total rows:", raw.count())
raw.printSchema()

# Preview data
display(raw.limit(10))

# Check duplicate neighbourhood names
duplicates = (
    raw.groupBy("neighbourhood")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate neighbourhoods:", duplicates.count())
display(duplicates)



StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 17, Finished, Available, Finished, False)

Total rows: 30
root
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 0e6c54bf-514d-46bf-bcb8-fe54f54f6093)

Duplicate neighbourhoods: 0


SynapseWidget(Synapse.DataFrame, 9f963365-6dd7-4397-984c-1d98a19837e8)

## **1. Data Quality Assessment**

### Missing Values and Duplicates

Before transforming the data, check for missing values
and duplicated neighbourhood names.

The dataset contains two columns:
- neighbourhood: Local Government Area name
- neighbourhood_group: Additional geographic grouping

The main focus is the neighbourhood column because
it will be used to link neighbourhoods with **Listings table**.

### Validation Criteria
- Each LGA should appear only once.
- LGA names must not be missing or blank.
- Standardised LGA keys must be unique.

In [16]:
# Check missing values in each column
missing = raw.select(*[
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == "") |
            (F.upper(F.trim(F.col(c))) == "NULL"),
            1
        ).otherwise(0)
    ).alias(c)
    for c in raw.columns
])

display(missing)


StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6812d6a5-a253-4515-b147-fbfe133b7297)

## **2. Data Cleaning and Standardisation**

### Transformation Logic

The neighbourhood column is renamed to `lga` to make
its business meaning clearer.

A new column, `lga_key`, is created by:
- Removing leading and trailing spaces
- Replacing multiple spaces with a single space
- Converting text to lowercase

### Why Standardisation Matters

Different text formats can cause JOIN mismatches.

For example:
- "Melbourne"
- " MELBOURNE "
- "melbourne"

All three values become `melbourne` in `lga_key`.

The original LGA name is retained for reporting,
while the standardised key is used for joining tables.

In [17]:
# Step 1: Select and clean LGA names
silver_neighbourhoods = raw.select(
    F.trim(F.col("neighbourhood")).alias("lga")
)

# Step 2: Convert empty strings and "NULL" text to NULL
silver_neighbourhoods = silver_neighbourhoods.withColumn(
    "lga",
    F.when(
        F.col("lga").isNull() |
        (F.col("lga") == "") |
        (F.upper(F.col("lga")) == "NULL"),
        F.lit(None).cast("string")
    ).otherwise(F.col("lga"))
)

# Step 3: Create standardised LGA join key
silver_neighbourhoods = silver_neighbourhoods.withColumn(
    "lga_key",
    F.lower(
        F.regexp_replace(
            F.col("lga"),
            r"\s+",
            " "
        )
    )
)

StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 19, Finished, Available, Finished, False)

## **3. Creating Silver Table**

After validation, the cleaned dataset is saved as a
Delta table named `dbo.silver_neighbourhoods`.

### Why Delta Format?
Delta Lake supports reliable table storage, schema
management and transactional data operations.

The table is only saved when the data quality
checks pass.

### Expected Output
- 30 LGAs
- Unique standardised LGA keys
- No missing LGA keys
- No duplicated LGA keys

In [18]:

# Validate and Save Silver Neighbourhoods

# 1. Calculate row counts
total = silver_neighbourhoods.count()
bronze_rows = raw.count()

# 2. Check unique LGA keys
unique_lga = (
    silver_neighbourhoods
    .select("lga_key")
    .distinct()
    .count()
)

# 3. Check missing values
missing_lga = (
    silver_neighbourhoods
    .filter(
        F.col("lga").isNull() |
        F.col("lga_key").isNull()
    )
    .count()
)

# 4. Check duplicate LGA keys
duplicate_count = (
    silver_neighbourhoods
    .groupBy("lga_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 5. Display validation results
print("Bronze Rows:", bronze_rows)
print("Total LGAs:", total)
print("Unique LGAs:", unique_lga)
print("Missing LGAs:", missing_lga)
print("Duplicate LGAs:", duplicate_count)

# 6. Dynamic Data Quality Validation
assert total > 0, "No LGAs found!"
assert total == bronze_rows, "Unexpected row loss!"
assert unique_lga == total, "LGA keys are not unique!"
assert missing_lga == 0, "Missing LGA keys!"
assert duplicate_count == 0, "Duplicate LGAs found!"

print("Silver Neighbourhood validation passed!")

# 7. Save Silver Table
(
    silver_neighbourhoods.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("dbo.silver_neighbourhoods")
)

print("Silver Neighbourhoods updated successfully!")


StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 20, Finished, Available, Finished, False)

Bronze Rows: 30
Total LGAs: 30
Unique LGAs: 30
Missing LGAs: 0
Duplicate LGAs: 0
Silver Neighbourhood validation passed!
Silver Neighbourhoods updated successfully!


## **4. Final Validation**

In [19]:
# Final Validation: Verify saved Silver Table

final_table = spark.table("dbo.silver_neighbourhoods")

saved_rows = final_table.count()

print("Saved rows:", saved_rows)
print("Saved columns:", final_table.columns)

assert saved_rows == total, \
    "Saved Silver row count does not match validated data!"

assert set(final_table.columns) == {"lga", "lga_key"}, \
    "Unexpected Silver table schema!"

print("Final Silver validation passed!")

display(final_table.orderBy("lga"))



StatementMeta(, 141690ea-fa04-46b1-910b-5f6a0a315171, 21, Finished, Available, Finished, False)

Saved rows: 30
Saved columns: ['lga', 'lga_key']
Final Silver validation passed!


SynapseWidget(Synapse.DataFrame, 8093fb96-b8b4-4b80-8763-cf740570bae6)